# Temporal Consistency Analysis

This notebook validates the temporal consistency of customer order history in the Instacart dataset.

The analysis focuses on order sequence, prior-order intervals, and chronological behavior before temporal features are created.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

orders = pd.read_csv(
    DATA_DIR / "orders.csv"
)

orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


## Order Sequence Consistency

In [3]:
order_sequence = (
    orders
    .sort_values(["user_id", "order_number"])
    .groupby("user_id")["order_number"]
)

sequence_summary = order_sequence.agg(
    min_order="min",
    max_order="max",
    unique_orders="nunique"
)

sequence_summary.head()

,min_order,max_order,unique_orders
user_id,,,
1,1,11,11
2,1,15,15
3,1,13,13
4,1,6,6
5,1,5,5


In [4]:
sequence_issues = 0

for user_id, group in orders.groupby("user_id"):
    order_numbers = sorted(group["order_number"].tolist())
    expected = list(range(1, len(order_numbers) + 1))

    if order_numbers != expected:
        sequence_issues += 1

print("Customers with inconsistent order sequences:", sequence_issues)

Customers with inconsistent order sequences: 0


## Prior-Order Interval Consistency

In [5]:
interval_check = orders[
    ["user_id", "order_number", "days_since_prior_order"]
].copy()

print(
    "Negative intervals:",
    (interval_check["days_since_prior_order"] < 0).sum()
)

print(
    "Intervals above 30 days:",
    (interval_check["days_since_prior_order"] > 30).sum()
)

Negative intervals: 0
Intervals above 30 days: 0


In [6]:
first_order_check = orders[orders["order_number"] == 1]

print(
    "First orders:",
    len(first_order_check)
)

print(
    "First orders with missing interval:",
    first_order_check["days_since_prior_order"].isna().sum()
)

print(
    "First orders with non-missing interval:",
    first_order_check["days_since_prior_order"].notna().sum()
)

First orders: 206209
First orders with missing interval: 206209
First orders with non-missing interval: 0


In [7]:
non_first_orders = orders[orders["order_number"] > 1]

print(
    "Non-first orders:",
    len(non_first_orders)
)

print(
    "Non-first orders with missing interval:",
    non_first_orders["days_since_prior_order"].isna().sum()
)

Non-first orders: 3214874
Non-first orders with missing interval: 0


In [8]:
temporal_summary = pd.DataFrame({
    "Check": [
        "Customers with inconsistent order sequences",
        "Negative prior-order intervals",
        "Intervals above 30 days",
        "First orders with missing interval",
        "First orders with non-missing interval",
        "Non-first orders with missing interval",
    ],
    "Count": [
        sequence_issues,
        int((orders["days_since_prior_order"] < 0).sum()),
        int((orders["days_since_prior_order"] > 30).sum()),
        int(first_order_check["days_since_prior_order"].isna().sum()),
        int(first_order_check["days_since_prior_order"].notna().sum()),
        int(non_first_orders["days_since_prior_order"].isna().sum()),
    ]
})

temporal_summary

,Check,Count
0,Customers with inconsistent order sequences,0
1,Negative prior-order intervals,0
2,Intervals above 30 days,0
3,First orders with missing interval,206209
4,First orders with non-missing interval,0
5,Non-first orders with missing interval,0
